In [ ]:
import asyncio
import random
import sys
import aiocsv
import aiofiles
import aiohttp
from aiohttp import ClientSession
from bs4 import BeautifulSoup, Tag
from datetime import date, datetime, time, timedelta
import logging
from tqdm import tqdm
from itertools import batched
# ================== НАСТРОЙКИ ==================

topics = {
    3: 'economics',
    4: 'business',
    2: 'politics',
    40: 'financies'
    
}

BASE_URL = "https://www.kommersant.ru/archive/rubric"
START_DATE = date(2022, 5, 1)
END_DATE = date.today()
CSV_FILE = "all_kommersant_news_final.csv"

MAX_CONCURRENT_REQUESTS = 5       # меньше одновременных подключений
ARTICLE_SEMAPHORE = asyncio.Semaphore(5)  # лимит для загрузки статей
REQUEST_TIMEOUT = 20
RETRY_COUNT = 5

DELAY_BETWEEN_ARTICLES = (0.3, 0.5)
DELAY_BETWEEN_DAYS = (0.5, 1.5)

USER_AGENTS = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/118.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 13_3) AppleWebKit/605.1.15 "
    "(KHTML, like Gecko) Version/16.3 Safari/605.1.15",
    "Mozilla/5.0 (X11; Linux x86_64) Gecko/20100101 Firefox/120.0",
]


# ================== ВСПОМОГАТЕЛЬНЫЕ ФУНКЦИИ ==================

def make_url(day: date, topic_id: int) -> str:
    return f"{BASE_URL}/{topic_id}/day/{day.strftime('%Y-%m-%d')}"

def daterange(start_date: date, end_date: date):
    """Возвращает все даты в диапазоне [start_date, end_date]."""
    days = (end_date - start_date).days
    return [start_date + timedelta(days=i) for i in range(days + 1)]

def make_dt(day: date, html: Tag):
    timetag = html.find("p", class_="uho__tag rubric_lenta__item_tag hide_desktop")
    if timetag:
        try:
            timetext = timetag.text.split(", ")[-1]
            hour, minute = map(int, timetext.split(":"))
            timestamp = time(hour=hour, minute=minute)
            return datetime.combine(day, timestamp)
        except Exception:
            return None

def parse_article_text(soup: BeautifulSoup) -> str:
    text = ""
    for p in soup.find_all("p", class_="doc__text"):
        text += p.text.strip() + " "
    return text.strip()

def get_article_cards(soup: BeautifulSoup):
    cards = []
    lenta_tags = soup.find_all("div", class_="rubric_lenta")
    for lenta_tag in lenta_tags:
        articles = lenta_tag.find_all("article")
        cards.extend(articles)
    return cards

# ================== АСИНХРОННЫЕ ФУНКЦИИ ==================

async def fetch(session: ClientSession, url: str, retries: int = RETRY_COUNT) -> str:
    """Асинхронно получает HTML страницы с повторами и экспоненциальной задержкой."""
    for attempt in range(retries):
        try:
            async with session.get(url, timeout=REQUEST_TIMEOUT) as response:
                if response.status == 200:
                    return await response.text()
                elif response.status in (429, 503, 502):
                    wait = 2 ** attempt + random.uniform(0.5, 1.5)
                    logging.warning(f"{response.status} при загрузке {url}, повтор через {wait:.1f} сек...")
                    await asyncio.sleep(wait)
                    continue
                else:
                    logging.warning(f"HTTP {response.status} при загрузке {url}")
                    return ""
        except (aiohttp.ClientError, asyncio.TimeoutError) as e:
            wait = 2 ** attempt + random.uniform(0.5, 1.5)
            logging.warning(f"Ошибка {type(e).__name__} при {url}, повтор через {wait:.1f} сек...")
            await asyncio.sleep(wait)
    logging.error(f"Не удалось загрузить {url} после {retries} попыток.")
    return ""


async def parse_article(session: ClientSession, day: date, card: Tag, topic_name: str):
    """Парсит одну статью с паузами и семафором."""
    async with ARTICLE_SEMAPHORE:
        url = card.get("data-article-url")
        heading = card.get("data-article-title", "").strip()

        if not url:
            return None

        await asyncio.sleep(random.uniform(*DELAY_BETWEEN_ARTICLES))  # небольшая пауза перед запросом

        html = await fetch(session, url)
        if not html:
            return None

        soup = BeautifulSoup(html, "html.parser")
        text = parse_article_text(soup)
        if not text:
            return None

        dt = make_dt(day, card)
        return [dt, topic_name, text, heading, url]



async def parse_news(session: ClientSession, days: list[date], sem: asyncio.Semaphore):
    """Парсит все статьи за один день."""
    async with sem:
        cards = {}
        for day in days:
            cards[day] = {}
            for topic_id, topic_name in topics.items():
                url = make_url(day, topic_id)
                html = await fetch(session, url)
                if not html:
                    continue

                soup = BeautifulSoup(html, "html.parser")
                cards[day][topic_name] = get_article_cards(soup)
                await asyncio.sleep(random.uniform(*DELAY_BETWEEN_DAYS))
            

        # tasks = [parse_article(session, day, card, topic_name) for card in cards]
        tasks = [parse_article(session, day, card, topic_name) for day in cards.keys() for topic_name in cards[day].keys() for card in cards[day][topic_name]]
        # results = await asyncio.gather(*tasks)
        # valid = [r for r in results if r]
        return tasks


async def write_csv_header():
    """Создает CSV с заголовками, если его нет."""
    try:
        async with aiofiles.open(CSV_FILE, "x", newline="", encoding="utf-8") as f:
            writer = aiocsv.AsyncWriter(f)
            await writer.writerow(["date", "topic", "text", "heading", "url"])
    except FileExistsError:
        pass  # файл уже существует


async def append_to_csv(rows: list[list]):
    """Добавляет данные в CSV."""
    if not rows:
        return
    async with aiofiles.open(CSV_FILE, "a", newline="", encoding="utf-8") as f:
        writer = aiocsv.AsyncWriter(f)
        await writer.writerows(rows)


# ================== ОСНОВНАЯ ЛОГИКА ==================

async def main():
    await write_csv_header()
    days = daterange(START_DATE, END_DATE)
    sem = asyncio.Semaphore(MAX_CONCURRENT_REQUESTS)
    headers = {
        "User-Agent": random.choice(USER_AGENTS),
        "Accept-Language": "ru,en;q=0.9",
        "Accept-Encoding": "gzip, deflate, br",
        "Connection": "keep-alive"
    }

    async with aiohttp.ClientSession(headers=headers) as session:
        print('Начинаем парсинг')
        for day in tqdm(days):
            period_tasks = await parse_news(session, [day], sem)
            rows = await asyncio.gather(*period_tasks)
            if len(rows) > 20:
                for batch in batched(rows, 5):
                    valid_rows = [r for r in batch if r]
                    await append_to_csv(valid_rows)
                    # await asyncio.sleep(0.2)
                continue
            valid_rows = [r for r in rows if r]
            await append_to_csv(valid_rows)
            # await asyncio.sleep(0.5)
            
            print(f"✅ {len(valid_rows)} статей сохранено")

    print("Парсинг завершён!")


await main()

Начинаем парсинг


  0%|          | 0/1275 [00:00<?, ?it/s]

  0%|          | 1/1275 [00:16<5:46:58, 16.34s/it]

✅ 14 статей сохранено


  0%|          | 3/1275 [00:52<6:09:49, 17.44s/it]

✅ 20 статей сохранено


  0%|          | 6/1275 [02:15<7:59:18, 22.66s/it]


CancelledError: 